# Silver Layer — Data Validation and Transformation

**Objective:** Transform raw fintech transactions into validated, typed, and deduplicated records using PySpark, and persist the results in a managed Delta Lake table.

### Why?

**For our project:** Prevents invalid and duplicate transactions from affecting financial KPIs in the Gold layer.

**For Data Engineering:** Demonstrates data type standardization, data quality enforcement, deduplication, and reliable Delta Lake persistence within the Medallion architecture.

#### 01 — Read and Inspect Bronze Data

Read the persisted Bronze transactions and verify their schema and initial record count. This establishes the input baseline before applying Silver transformations.

In [0]:
df_bronze = spark.table("workspace.default.fintech_transactions_bronze")
display(df_bronze)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
df_bronze.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
assert df_bronze.count() == 12, "Unexpected Bronze row count"

#### 02 — Convert and Validate Data Types

Convert `amount` to DECIMAL(12,2) and `transaction_ts` to TIMESTAMP. These conversions enable accurate financial calculations and date-based processing.

Verify the resulting schema and confirm that type conversion preserves the original record count.

In [0]:
import pyspark.sql.functions as sf

# Use fixed decimal precision for financial amounts
df_silver_typed = df_bronze.withColumn("amount", sf.col("amount").cast("decimal(12,2)"))    
display(df_silver_typed)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
import pyspark.sql.functions as sf

# Convert transaction timestamps for date-based analysis
df_silver_typed = df_silver_typed.withColumn("transaction_ts", sf.to_timestamp(sf.col("transaction_ts"), "yyyy-MM-dd HH:mm:ss"))    
display(df_silver_typed)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18T14:15:00.000Z,CARD_PURCHASE,25.00,COMPLETED


In [0]:
df_silver_typed.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)



In [0]:
assert df_silver_typed.count() == 12, "Unexpected row count"

In [0]:
display(df_silver_typed)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18T14:15:00.000Z,CARD_PURCHASE,25.00,COMPLETED


#### 03 — Apply Data Quality Rules

Apply six validation rules to transaction identifiers, amounts, timestamps, transaction types, and statuses. Separate valid and invalid records to prevent unreliable data from reaching the Silver table.

Use `exceptAll()` to preserve row multiplicities during validation and verify that valid and invalid records account for the complete input dataset.

##### DQR-001/002:

Reject missing transaction or customer identifiers

In [0]:
import pyspark.sql.functions as sf

df_invalid_ids = df_silver_typed.filter( sf.col("transaction_id").isNull() | sf.col("customer_id").isNull() )
display(df_invalid_ids)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
null,C007,2026-10-18T14:15:00.000Z,CARD_PURCHASE,25.00,COMPLETED


In [0]:
assert df_invalid_ids.count() == 1, "Unexpected invalid identifier count"

##### DQR-003:

Reject null, zero, or negative amounts

In [0]:
import pyspark.sql.functions as sf

df_invalid_amounts = df_silver_typed.filter( sf.col("amount").isNull() | (sf.col("amount") <= 0) )
display(df_invalid_amounts)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX009,C008,2026-10-18T15:00:00.000Z,PIX_IN,-50.00,COMPLETED
TX010,C009,2026-10-18T16:00:00.000Z,PIX_IN,null,COMPLETED


In [0]:
assert df_invalid_amounts.count() == 2, "Unexpected invalid amount count"

##### DQR-004:

Reject invalid transaction timestamps

In [0]:
import pyspark.sql.functions as sf

df_invalid_timestamps = df_silver_typed.filter( sf.col("transaction_ts").isNull())
display(df_invalid_timestamps)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_timestamps.count() == 0, "Unexpected invalid transaction_ts count"

##### DQR-005:

Reject unsupported transaction types

In [0]:
import pyspark.sql.functions as sf

df_invalid_transaction_types = df_silver_typed.filter( sf.col("transaction_type").isNull() | ~ ( sf.col("transaction_type").isin(["PIX_IN", "PIX_OUT", "CARD_PURCHASE", "BILL_PAYMENT"]) ) )
display(df_invalid_transaction_types)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_transaction_types.count() == 0, "Unexpected invalid transaction type count"

##### DQR-006:

Reject unsupported transaction statuses

In [0]:
import pyspark.sql.functions as sf

df_invalid_statuses = df_silver_typed.filter( sf.col("status").isNull() | ~ ( sf.col("status").isin(["COMPLETED", "PENDING", "FAILED"]) ) )
display(df_invalid_statuses)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status


In [0]:
assert df_invalid_statuses.count() == 0, "Unexpected invalid status type count"

#### Consolidate data quality rules

In [0]:
# Remove invalid records while preserving duplicate row counts
df_silver_valid = df_silver_typed

for df_invalid in [
    df_invalid_ids,
    df_invalid_amounts,
    df_invalid_timestamps,
    df_invalid_transaction_types,
    df_invalid_statuses
]:
    df_silver_valid = df_silver_valid.exceptAll(df_invalid)

df_silver_invalid = df_silver_typed.exceptAll(df_silver_valid)

In [0]:
# Verify valid and invalid record counts and ensure no records were lost during validation
assert df_silver_valid.count() == 9
assert df_silver_invalid.count() == 3

assert (
    df_silver_valid.count() + df_silver_invalid.count()
    == df_silver_typed.count()
)

In [0]:
display(df_silver_valid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
display(df_silver_invalid)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
null,C007,2026-10-18T14:15:00.000Z,CARD_PURCHASE,25.00,COMPLETED
TX010,C009,2026-10-18T16:00:00.000Z,PIX_IN,null,COMPLETED
TX009,C008,2026-10-18T15:00:00.000Z,PIX_IN,-50.00,COMPLETED


#### 04 — Deduplicate Valid Transactions

Remove exact duplicate records from the valid transactions using `dropDuplicates()`. This prevents repeated source records from being counted more than once in downstream financial metrics.

Validate that the initial dataset contains eight unique transactions after deduplication.

In [0]:
df_silver_deduplicated = df_silver_valid.dropDuplicates()
display(df_silver_deduplicated)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
assert df_silver_deduplicated.count() == 8, "Unexpected deduplicated row count"

#### 05 — Persist and Validate Silver Data

Persist the eight validated and deduplicated transactions in a managed Delta Lake table.

Read the persisted data and verify record counts, schema, unique transaction IDs, and Delta transaction history. These checks confirm the integrity of the initial Silver dataset.

In [0]:
# Create the initial Silver table without replacing existing data
df_silver_deduplicated.write.format("delta").mode("error").saveAsTable("workspace.default.fintech_transactions_silver")

In [0]:
# Read and inspect the persisted Silver Delta table to verify that validated transactions are available for downstream processing
df_silver_saved = spark.table("workspace.default.fintech_transactions_silver")
display(df_silver_saved)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
df_silver_saved.count()

8

In [0]:
df_silver_saved.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)



In [0]:
%sql
-- Inspect Delta transaction history to verify table operations and support data traceability and auditing
DESCRIBE HISTORY workspace.default.fintech_transactions_silver;

In [0]:
# Bronze table remains unchanged with 12 records
df_bronze_saved = spark.table("workspace.default.fintech_transactions_bronze")
display(df_bronze_saved)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
# Validate Silver row count, unique transaction IDs, and preservation of the original Bronze records
assert df_silver_saved.count() == 8, "Unexpected Silver row count"

assert (
    df_silver_saved.select("transaction_id").distinct().count()
    == df_silver_saved.count()
), "Duplicate transaction IDs found"

assert df_bronze_saved.count() == 12, "Unexpected Bronze row count"